# 00 - Audit of the existing cache (CPU only, safe to run locally)

What this notebook does: inspects `results/cached_features/pope_subset_features.pt` and the local POPE dataset, and tests whether the cached tensors can be tied to dataset rows. It loads **no model** and needs **no GPU**.

Requirements locally: `torch`, `numpy`, `pandas`, `matplotlib`, and `datasets` (only for the dataset section). `transformers` is optional (used only to read the model config if it can be fetched).

In [ ]:
import os, sys, collections
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib
import matplotlib.pyplot as plt

def find_repo_root(start=None):
    p = Path(start or os.getcwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "src" / "download_datasets.py").exists():
            return q
    raise RuntimeError("Run this notebook from inside the repo")

ROOT = find_repo_root()
os.chdir(ROOT)                                   # src/download_datasets.py uses the relative path data/pope
sys.path.insert(0, str(ROOT / "src"))
CACHE_PATH = ROOT / "results" / "cached_features" / "pope_subset_features.pt"
FIG_DIR = Path(os.environ.get("POPE_FIG_DIR", ROOT / "results" / "figures")); FIG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"           # as hardcoded in src/feature_extraction.py
print("repo root:", ROOT)

## 1. What is in the cache file?

In [ ]:
cache = torch.load(CACHE_PATH, map_location="cpu")
print("file size: %.2f MB" % (CACHE_PATH.stat().st_size / 1e6))
print("python type:", type(cache).__name__, "| number of samples:", len(cache))
shapes = collections.Counter(tuple(t.shape) for t in cache)
dtypes = collections.Counter(str(t.dtype) for t in cache)
print("per-sample tensor shapes:", dict(shapes), "| dtypes:", dict(dtypes))
N_CACHED = len(cache)
N_LAYERS_CACHE, HIDDEN_CACHE = cache[0].shape            # read from the file, not hardcoded
print(f"hidden-state entries per sample (dim 0): {N_LAYERS_CACHE} | hidden size (dim 1): {HIDDEN_CACHE}")
X = torch.stack(cache).float()                            # [N, L, H]
print("all finite:", bool(torch.isfinite(X).all()), "| number of exactly duplicated samples:",
      N_CACHED - len({x.numpy().tobytes() for x in X.half()}))
print("std across samples of hidden-state entry 0 (embedding of the last prompt token): %.4g" % X[:, 0].std(0).max().item(),
      "-> entry 0 is the same token embedding for every prompt, so it carries no per-question information" if X[:, 0].std(0).max() < 1e-3 else "")
print("Keys/labels/predictions stored in file: none - the file is a bare list of tensors.")

Compare with the model config if it is reachable (needs internet + `transformers`; otherwise this cell just says so):

In [ ]:
try:
    from transformers import AutoConfig
    cfg = AutoConfig.from_pretrained(MODEL_ID)
    tcfg = cfg.get_text_config() if hasattr(cfg, "get_text_config") else cfg
    n_hidden_layers, hidden_size = tcfg.num_hidden_layers, tcfg.hidden_size
    print(f"model config: num_hidden_layers={n_hidden_layers}, hidden_size={hidden_size}")
    print("cache entries == num_hidden_layers + 1 (embeddings + layers)?", N_LAYERS_CACHE == n_hidden_layers + 1,
          "| hidden size matches?", HIDDEN_CACHE == hidden_size)
except Exception as e:
    print("Model config not available here (", type(e).__name__, "). Layer count above is read from the cache file only;"
          " by HF convention it is num_hidden_layers + 1 (embedding output + one per layer).")

## 2. The POPE dataset as it actually is

Loaded through `src/download_datasets.py`. If `data/pope/` is not present locally, this section does not download anything by itself: run `python src/download_datasets.py` once (about 255 MB), or skip it, since the Colab notebook downloads the dataset itself.

In [ ]:
DATA_PATH = ROOT / "data" / "pope"
pope, df = None, None
if DATA_PATH.exists() and any(DATA_PATH.iterdir()):
    from download_datasets import load_or_download_pope      # finds data/pope and loads from disk (no download)
    pope = load_or_download_pope()
else:
    print("data/pope/ is missing or empty -> dataset section skipped. Run `python src/download_datasets.py` locally,"
          " or rely on the download step in notebook 01 (Colab).")

In [ ]:
if pope is not None:
    print("split names:", list(pope.keys()))
    for name, d in pope.items():
        print(f"\n[{name}] rows={len(d)} fields={list(d.features.keys())}")
    split = "test" if "test" in pope else list(pope.keys())[0]     # same split feature_extraction.py hardcodes
    d = pope[split]
    df = d.remove_columns("image").to_pandas()
    df["row_idx"] = np.arange(len(df))
    df["answer"] = df["answer"].str.strip().str.lower()
    print(f"\nusing split '{split}': {len(df)} rows")
    print("yes/no balance:", df["answer"].value_counts().to_dict())
    has_cat = "category" in df.columns
    print("\nrandom / popular / adversarial are:",
          "SEPARATE SPLITS" if {"random", "popular", "adversarial"} <= set(pope.keys()) else
          ("a FIELD called 'category' inside one split" if has_cat else "NOT PRESENT"))
    if has_cat:
        print(pd.crosstab(df["category"], df["answer"]))
    print("unique images (image_source):", df["image_source"].nunique(),
          "| images that appear in more than one setting:", int((df.groupby("image_source")["category"].nunique() > 1).sum()))
    print("'id' unique:", df["id"].is_unique, "| 'question_id' unique:", df["question_id"].is_unique,
          "(question_id restarts per setting)")
    print("\nfirst rows:"); display(df.head(5)[["row_idx", "id", "question_id", "question", "answer", "image_source", "category"]])

## 3. Do the 100 cached tensors align with the first 100 dataset rows?

`src/feature_extraction.py` iterates `range(min(100, len(split)))` over split `"test"` in dataset order and appends one tensor per row without shuffling, so *if the script was run unchanged on this dataset version*, tensor `i` belongs to row `i`. The file itself carries no row ids, labels or hashes, so this **cannot be proven from the file**. What can be checked:

In [ ]:
ALIGNMENT_VERIFIED = False       # stays False: nothing in the cache file identifies which rows it came from
checks = []
checks.append(("cache has 100 tensors (script's subset_size)", N_CACHED == 100))
if df is not None:
    first = df.iloc[:N_CACHED]
    checks.append(("dataset has at least that many rows", len(df) >= N_CACHED))
    print("First %d dataset rows: settings = %s | yes/no = %s | distinct images = %d" % (
        N_CACHED, first["category"].value_counts().to_dict(), first["answer"].value_counts().to_dict(), first["image_source"].nunique()))
for desc, ok in checks:
    print(("PASS " if ok else "FAIL "), desc)
print("\nVERDICT: counts are consistent, but alignment to rows 0..99 is NOT verifiable from the file.")
print("-> Treat pope_subset_features.pt as EXPLORATORY ONLY. If it is aligned, it covers only the first 100 rows,"
      " which (in this dataset version) are all from the 'adversarial' setting, so it is not representative of POPE.")
print("-> Notebook 02 has an optional check that compares it with the first 100 rows of the new full cache, which would settle this.")

## 4. Unsupervised exploratory plots  —  NOT hallucination detection
The current cache has **no model predictions**, so nothing below says anything about hallucination. These are just hidden-state norms.

In [ ]:
norms = X.norm(dim=-1).numpy()                  # [N, L]
fig, ax = plt.subplots(figsize=(7, 3.8))
layers = np.arange(N_LAYERS_CACHE)
m, s = norms.mean(0), norms.std(0)
ax.plot(layers, m, marker="o", ms=3)
ax.fill_between(layers, m - s, m + s, alpha=0.25)
ax.set_yscale("log"); ax.set_xlabel("hidden-state index (0 = embedding output)"); ax.set_ylabel("L2 norm of last-prompt-token state")
ax.set_title("Hidden-state norm by layer, 100 cached samples\n(exploratory, NOT hallucination detection)")
fig.tight_layout(); fig.savefig(FIG_DIR / "audit_norm_by_layer.png", dpi=150); plt.show()

Norm by ground-truth present (`yes`) vs absent (`no`). This **assumes** tensor `i` = dataset row `i`, which is unverified (section 3), so it only runs if you opt in by setting the flag below. Even then it is exploratory and not hallucination detection.

In [ ]:
ASSUME_ALIGNED_FOR_EXPLORATION = False          # opt-in
if ASSUME_ALIGNED_FOR_EXPLORATION and df is not None:
    gt = df["answer"].iloc[:N_CACHED].to_numpy()
    fig, ax = plt.subplots(figsize=(7, 3.8))
    for lab, c in [("yes", "tab:green"), ("no", "tab:red")]:
        mm = norms[gt == lab].mean(0)
        ax.plot(layers, mm, label=f"ground truth '{lab}' (object {'present' if lab == 'yes' else 'absent'}), n={(gt == lab).sum()}", color=c)
    ax.set_yscale("log"); ax.legend(); ax.set_xlabel("hidden-state index"); ax.set_ylabel("mean L2 norm")
    ax.set_title("Norm by ground truth (ASSUMES unverified alignment; NOT hallucination detection)")
    fig.tight_layout(); fig.savefig(FIG_DIR / "audit_norm_by_ground_truth.png", dpi=150); plt.show()
else:
    print("Skipped: alignment unverified (ASSUME_ALIGNED_FOR_EXPLORATION=False) or dataset not available locally.")